# 03 — Walk-Forward Evaluation

Anchored expanding train window of 5 years; 1-year non-overlapping OOS test windows; re-optimise every year by best in-sample Sharpe with a neighbourhood-stability tie-break.

The concatenated OOS equity curve is the *only* honest performance estimate this project produces.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from dataclasses import asdict

from ma_backtester.backtester import run_buy_and_hold
from ma_backtester.config import DEFAULT_SWEEP, CostConfig, WalkForwardConfig
from ma_backtester.costs import FixedBpsCost
from ma_backtester.data import load_close
from ma_backtester.data_snooping import deflated_sharpe_ratio
from ma_backtester.metrics import sharpe_ratio
from ma_backtester.plotting import equity_curve, install_theme
from ma_backtester.walk_forward import run_walk_forward

install_theme()

In [ ]:
ticker = 'SPY'
close = load_close(ticker, start='2005-01-01', end='2024-12-31')
cost = FixedBpsCost(CostConfig(per_side_bps=5.0))
wf_config = WalkForwardConfig(train_years=5, test_years=1, step_years=1)

## Run walk-forward

In [ ]:
wf = run_walk_forward(
    close=close, ticker=ticker, sweep=DEFAULT_SWEEP,
    wf_config=wf_config, cost_model=cost,
)
print(f'{len(wf.folds)} OOS folds.')

## Per-fold table — selected params, in-sample vs out-of-sample

In [ ]:
fold_df = pd.DataFrame([{
    'fold': f.fold_index,
    'train_end': f.train_end.date(),
    'test_end': f.test_end.date(),
    'fast': f.selected_fast,
    'slow': f.selected_slow,
    'IS_sharpe': f.in_sample_sharpe,
    'OOS_sharpe': f.out_of_sample_sharpe,
    'OOS_return': f.out_of_sample_return,
} for f in wf.folds])
fold_df

## In-sample vs out-of-sample Sharpe — the most diagnostic chart

If the strategy generalises, points should lie near the y=x line. A flat or negative slope means in-sample Sharpe doesn't predict OOS — i.e. we're overfitting.

In [ ]:
fig = go.Figure()
fig.add_scatter(
    x=fold_df['IS_sharpe'], y=fold_df['OOS_sharpe'],
    mode='markers+text', text=fold_df['fold'].astype(str),
    textposition='top right',
)
lim = float(max(fold_df['IS_sharpe'].abs().max(), fold_df['OOS_sharpe'].abs().max()) + 0.2)
fig.add_shape(type='line', x0=-lim, y0=-lim, x1=lim, y1=lim, line=dict(dash='dash'))
fig.update_layout(
    title='In-sample vs OOS Sharpe (one point per fold)',
    xaxis_title='In-sample Sharpe', yaxis_title='OOS Sharpe',
)
fig

## Parameter stability across folds

In [ ]:
fig = go.Figure()
fig.add_scatter(x=fold_df['fold'], y=fold_df['fast'], mode='lines+markers', name='fast')
fig.add_scatter(x=fold_df['fold'], y=fold_df['slow'], mode='lines+markers', name='slow')
fig.update_layout(title='Selected (fast, slow) per fold', xaxis_title='fold', yaxis_title='window (bars)')
fig

## Concatenated OOS equity vs buy-and-hold

In [ ]:
bench = run_buy_and_hold(close=close, cost_model=cost)
bench_oos = bench.equity.loc[wf.concatenated_equity.index]
bench_oos = bench_oos / bench_oos.iloc[0] * float(wf.concatenated_equity.iloc[0])
equity_curve(
    strategy_equity=wf.concatenated_equity,
    benchmark_equity=bench_oos,
    title='Concatenated out-of-sample equity vs B&H',
)

## DSR on the concatenated OOS series

In [ ]:
dsr = deflated_sharpe_ratio(
    daily_returns=wf.concatenated_returns,
    n_trials=DEFAULT_SWEEP.size,
)
pd.Series(asdict(dsr))